# Clase 163 — Markov Decision Processes (numpy, ejecutable)

Un **MDP** es la tupla `(S, A, P, R, γ)`. La **ecuación de Bellman** define el valor óptimo
`V*(s) = max_a Σ_s' P(s'|s,a)·[R(s,a,s') + γ·V*(s')]`.

Aquí construimos a mano un **gridworld resbaladizo 4×4** (estilo FrozenLake) con su matriz de
transición `P` y recompensa `R`, y lo resolvemos con **Value Iteration** y **Policy Iteration**
en numpy puro. Todo se ejecuta.

Requiere: `numpy`.

## 💡 Intuición previa

Un **MDP** (Markov Decision Process) es el tablero formal del aprendizaje por refuerzo: hay **estados** (dónde está el agente), **acciones** (qué puede hacer) y **recompensas** (qué tan bueno fue el resultado). El agente busca una **política** — una regla que dice qué acción tomar en cada estado — que **maximice la recompensa futura acumulada**, no solo la inmediata. Piénsalo como un GPS: cada intersección es un estado, cada giro una acción, y quieres la ruta que minimiza el tiempo total (recompensa negativa = tiempo), no solo el próximo giro. **Value Iteration** y **Policy Iteration** calculan esa ruta óptima cuando conoces el mapa (`P` y `R`).

## 1. Definir el MDP a mano: gridworld 4×4 resbaladizo

Grid (S=start, H=hole, G=goal):
```
 0(S)  1     2     3
 4     5(H)  6     7(H)
 8     9    10    11(H)
12(H) 13    14    15(G)
```
Acciones: 0=izq, 1=abajo, 2=der, 3=arriba. Es **resbaladizo**: la acción elegida se ejecuta
con prob 1/3, y con 1/3 cada una de las dos perpendiculares.

In [ ]:
import numpy as np
np.random.seed(0)

nS, nA = 16, 4
holes = {5, 7, 11, 12}
goal = 15

def next_state(s, a):
    row, col = divmod(s, 4)
    if   a == 0: col = max(col - 1, 0)     # izquierda
    elif a == 1: row = min(row + 1, 3)     # abajo
    elif a == 2: col = min(col + 1, 3)     # derecha
    elif a == 3: row = max(row - 1, 0)     # arriba
    return row * 4 + col

# acciones que ocurren al "resbalar": la intencion + las dos perpendiculares
perp = {0: (0, 1, 3), 1: (1, 0, 2), 2: (2, 1, 3), 3: (3, 0, 2)}

P = np.zeros((nS, nA, nS))                  # P[s, a, s']
R = np.zeros((nS, nA, nS))                  # R[s, a, s']
for s in range(nS):
    for a in range(nA):
        if s in holes or s == goal:
            P[s, a, s] = 1.0                # estados terminales: absorbentes
            continue
        for a_real in perp[a]:
            s2 = next_state(s, a_real)
            P[s, a, s2] += 1.0 / 3.0
            if s2 == goal:
                R[s, a, s2] = 1.0           # +1 solo al alcanzar la meta

assert np.allclose(P.sum(axis=2), 1.0)      # cada (s,a) es una distribucion valida
print("MDP construido: P", P.shape, "| suma de probabilidades OK")

## 2. Value Iteration

Aplicamos el operador de Bellman óptimo hasta que `V` deja de cambiar. La convergencia está
garantizada porque es una contracción (factor `γ < 1`).

In [ ]:
def value_iteration(P, R, gamma=0.99, tol=1e-10):
    V = np.zeros(nS)
    for it in range(1, 10001):
        # Q[s,a] = sum_s' P[s,a,s'] * (R[s,a,s'] + gamma * V[s'])
        Q = np.einsum("sap,sap->sa", P, R + gamma * V[None, None, :])
        V_new = Q.max(axis=1)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, Q.argmax(axis=1), it
        V = V_new
    return V, Q.argmax(axis=1), it

gamma = 0.99
V_star, pi_star, iters = value_iteration(P, R, gamma)
print(f"Value Iteration convergio en {iters} iteraciones")
print("V*  =\n", np.round(V_star.reshape(4, 4), 3))
arrows = np.array(list("<v>^"))
grid_pi = arrows[pi_star].reshape(4, 4)
for s in holes: grid_pi.flat[s] = "H"
grid_pi.flat[goal] = "G"
print("policy* =\n", grid_pi)

## 3. Policy Iteration

Alterna **evaluación** (resolver `V^π`) y **mejora** (`π' = greedy(V^π)`) hasta que la policy
se estabiliza. Suele necesitar menos iteraciones que Value Iteration, aunque cada una es más cara.

In [ ]:
def policy_evaluation(pi, P, R, gamma, tol=1e-10):
    V = np.zeros(nS)
    while True:
        # solo la accion elegida por la policy en cada estado
        P_pi = P[np.arange(nS), pi]                 # (nS, nS)
        R_pi = (P_pi * R[np.arange(nS), pi]).sum(axis=1)
        V_new = R_pi + gamma * (P_pi @ V)
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new

def policy_iteration(P, R, gamma):
    pi = np.zeros(nS, dtype=int)
    for it in range(1, 1001):
        V = policy_evaluation(pi, P, R, gamma)
        Q = np.einsum("sap,sap->sa", P, R + gamma * V[None, None, :])
        pi_new = Q.argmax(axis=1)
        if np.array_equal(pi_new, pi):
            return V, pi_new, it
        pi = pi_new
    return V, pi, it

V_pi, pi_pi, iters_pi = policy_iteration(P, R, gamma)
print(f"Policy Iteration convergio en {iters_pi} iteraciones")
print("misma policy que Value Iteration:", np.array_equal(pi_pi, pi_star))
print("misma V* (aprox):", np.allclose(V_pi, V_star, atol=1e-6))

## 4. Evaluar la policy con simulación Monte Carlo

Simulamos episodios muestreando el próximo estado desde `P` y medimos el **success rate**
(fracción de episodios que alcanzan la meta).

In [ ]:
def rollout(pi, P, max_steps=100, seed=0):
    rng = np.random.default_rng(seed)
    s = 0                                            # start
    for _ in range(max_steps):
        s = rng.choice(nS, p=P[s, pi[s]])
        if s == goal:  return True
        if s in holes: return False
    return False

n = 2000
success = np.mean([rollout(pi_star, P, seed=i) for i in range(n)])
print(f"success rate de la policy optima ({n} episodios): {success:.3f}")
print("(en FrozenLake resbaladizo, una buena policy logra >= 0.70)")

## 5. Limitación y motivación

Value/Policy Iteration **requieren conocer `P` y `R`** (el modelo del MDP) y no escalan a
espacios de estados grandes o continuos. En entornos reales rara vez conocemos `P`: eso motiva
los métodos **model-free** como Q-learning (clase 164).

## 6. Variante determinista (ejecutable)

Si el grid NO resbala, cada `(s,a)` lleva con certeza a un único estado. La policy óptima traza
el camino más corto a la meta evitando huecos, y el success rate llega a 1.0.

In [ ]:
P_det = np.zeros((nS, nA, nS))
R_det = np.zeros((nS, nA, nS))
for s in range(nS):
    for a in range(nA):
        if s in holes or s == goal:
            P_det[s, a, s] = 1.0
            continue
        s2 = next_state(s, a)                # sin resbalar: accion determinista
        P_det[s, a, s2] = 1.0
        if s2 == goal:
            R_det[s, a, s2] = 1.0

V_det, pi_det, it_det = value_iteration(P_det, R_det, gamma)
print(f"determinista: VI convergio en {it_det} iteraciones")
sr = np.mean([rollout(pi_det, P_det, seed=i) for i in range(500)])
print(f"success rate (determinista): {sr:.3f}")

## Ejercicios

1. Cambiar `γ` a 0.9 y 0.999; observar cómo se modifican `V*` y la policy.
2. Volver el gridworld **determinista** (`P[s,a,next_state(s,a)] = 1`) y verificar que la policy
   traza el camino más corto evitando huecos.
3. Añadir una penalización `-0.01` por paso en `R` y comprobar que la policy se vuelve más directa.
4. Comparar el número de iteraciones de Value Iteration vs Policy Iteration al variar `γ`.

## Conclusiones

- Un MDP `(S, A, P, R, γ)` formaliza el problema de RL cuando el modelo es conocido.
- Value Iteration aplica el operador de Bellman óptimo hasta converger (es una contracción).
- Policy Iteration alterna evaluación y mejora; converge en pocas iteraciones a la misma `π*`.
- Ambos exigen conocer `P` y `R`, lo que casi nunca ocurre: de ahí los métodos model-free.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Las de **núcleo numérico** son ejecutables (con `assert` de verificación); las de frameworks/servicios no instalados aquí (TF, PyTorch, diffusers, Gymnasium, GCP…) se muestran como **código real de referencia** listo para copiar en un entorno con esas dependencias.

### Ejercicio 1 — MDP de juguete (4 estados) definido a mano

In [ ]:
import numpy as np

# 4 estados en linea: 0 -> 1 -> 2 -> 3(meta). 2 acciones: 0=izquierda, 1=derecha.
nS, nA = 4, 2
P = np.zeros((nS, nA, nS))   # P[s, a, s']
R = np.zeros((nS, nA, nS))   # recompensa
for s in range(nS):
    left = max(s - 1, 0)
    right = min(s + 1, nS - 1)
    if s == 3:                 # estado meta: absorbente
        P[s, :, s] = 1.0
        continue
    P[s, 0, left] = 1.0
    P[s, 1, right] = 1.0
    if right == 3:
        R[s, 1, right] = 1.0   # +1 al llegar a la meta

assert np.allclose(P.sum(axis=2), 1.0), 'cada (s,a) es una distribucion de probabilidad'
print('MDP 4 estados definido; P y R validos')

### Ejercicio 2 — Value Iteration hasta `max change < 1e-6`

In [ ]:
def value_iteration(P, R, gamma=0.99, tol=1e-6):
    nS, nA, _ = P.shape
    V = np.zeros(nS); it = 0
    while True:
        Q = np.einsum('sap,sap->sa', P, R + gamma * V[None, None, :])
        V_new = Q.max(axis=1); it += 1
        if np.abs(V_new - V).max() < tol:
            V = V_new; break
        V = V_new
    pi = np.einsum('sap,sap->sa', P, R + gamma * V[None, None, :]).argmax(axis=1)
    return V, pi, it

V, pi, it = value_iteration(P, R)
print('V* =', np.round(V, 3), '| pi* =', pi, '| iters =', it)
assert (pi[:3] == 1).all(), 'la politica optima siempre va a la derecha (hacia la meta)'
assert V[0] > 0
print('OK: Value Iteration converge a la politica optima')

### Ejercicio 3 — Policy Iteration (evaluación + mejora)

In [ ]:
def policy_eval(pi, P, R, gamma, tol=1e-10):
    nS = P.shape[0]; V = np.zeros(nS)
    while True:
        Vn = np.array([P[s, pi[s]] @ (R[s, pi[s]] + gamma * V) for s in range(nS)])
        if np.abs(Vn - V).max() < tol:
            return Vn
        V = Vn

def policy_iteration(P, R, gamma=0.99):
    nS, nA, _ = P.shape
    pi = np.zeros(nS, dtype=int); it = 0
    while True:
        V = policy_eval(pi, P, R, gamma); it += 1
        Q = np.einsum('sap,sap->sa', P, R + gamma * V[None, None, :])
        pi_new = Q.argmax(axis=1)
        if (pi_new == pi).all():
            return V, pi_new, it
        pi = pi_new

V_pi, pi_pi, it_pi = policy_iteration(P, R)
assert (pi_pi == pi).all(), 'PI y VI encuentran la MISMA politica optima'
print('PI politica =', pi_pi, '| iteraciones de politica =', it_pi)
print('OK: Policy Iteration coincide con Value Iteration')

### Ejercicio 4 — FrozenLake: extraer el modelo y resolver con VI

En Gymnasium el modelo del MDP está en `env.unwrapped.P`. Como aquí no hay `gymnasium` instalado, **reconstruimos a mano** el FrozenLake 4×4 determinista (mapa oficial `SFFF/FHFH/FFFH/HFFG`) y lo resolvemos con la misma `value_iteration`.

In [ ]:
def build_frozenlake():
    desc = ['SFFF', 'FHFH', 'FFFH', 'HFFG']
    nS, nA = 16, 4                       # acciones: 0=L,1=D,2=R,3=U
    holes, goal = set(), None
    for r in range(4):
        for c in range(4):
            ch = desc[r][c]
            if ch == 'H': holes.add(r * 4 + c)
            if ch == 'G': goal = r * 4 + c
    P = np.zeros((nS, nA, nS)); R = np.zeros((nS, nA, nS))
    def mv(r, c, a):
        if a == 0: c = max(c - 1, 0)
        elif a == 1: r = min(r + 1, 3)
        elif a == 2: c = min(c + 1, 3)
        else: r = max(r - 1, 0)
        return r, c
    terminal = holes | {goal}
    for r in range(4):
        for c in range(4):
            s = r * 4 + c
            for a in range(4):
                if s in terminal:
                    P[s, a, s] = 1.0
                else:
                    nr, nc = mv(r, c, a); ns = nr * 4 + nc
                    P[s, a, ns] = 1.0
                    if ns == goal: R[s, a, ns] = 1.0
    return P, R, goal, holes

Pf, Rf, goal, holes = build_frozenlake()
Vf, pif, itf = value_iteration(Pf, Rf, gamma=0.99)

# Rollout greedy desde el inicio: debe llegar a la meta esquivando huecos
s, steps = 0, 0
while s != goal and steps < 50:
    s = Pf[s, pif[s]].argmax(); steps += 1
assert s == goal, 'la politica optima alcanza la meta'
assert not (set(range(16)) & holes & {s}), 'sin caer en huecos'
print('FrozenLake resuelto: V*[start]=%.3f, llega a la meta en %d pasos' % (Vf[0], steps))

Con Gymnasium instalado, el modelo se extrae directamente (sin reconstruirlo):

```python
import gymnasium as gym
env = gym.make('FrozenLake-v1', is_slippery=True)
model = env.unwrapped.P            # dict[s][a] -> [(prob, s', reward, done), ...]
# Convertir model a matrices P, R y llamar value_iteration(P, R)
```

### Ejercicio 5 — Comparar # de iteraciones VI vs PI

In [ ]:
_, pi_vi, it_vi = value_iteration(Pf, Rf)
_, pi_pi, it_pi = policy_iteration(Pf, Rf)
print('VI: %d barridos de valor | PI: %d iteraciones de politica' % (it_vi, it_pi))
# PI necesita muchas MENOS iteraciones (cada una es cara: resuelve V^pi completo)
assert it_pi <= it_vi
print('OK: PI converge en menos iteraciones (aunque cada una es mas costosa)')